# Mellum R1 LoRA — A6000 48GB

Fine-tunes `JetBrains/Mellum2.1-12B-A2.5B-Thinking` @ `92ddae9f` on the Wisp R1 tool-calling dataset.
Plan: `docs/plans/2026-10-14-mellum-finetune.md`. Run the cells top to bottom; each one is a `run.sh` step.

**Before this notebook:** paste `training/mellum-r1-train/vm-setup.sh` into the VM's setup-script field and boot.
Setup installs everything and downloads the 24GB base (~15–30 min).

**Memory plan (Mellum is 12.15B, 22.6 GiB bf16, vs Ling's ~16 GiB):**
- Loss is on the completion only, so logits are computed for those positions alone (`logits_to_keep`), not
  12k × 98k. This was the largest transient on Ling (~10 GiB fp32 logits + their gradient).
- `--max-len 12288`: the real longest row is ~11.1k tokens, so the 16k Ling cap only bought slack.
- LoRA on attention only (r=16): 8.3M trainable params, ~0.1 GiB with Adam state.
- Gradient checkpointing (non-reentrant), micro-batch 1, `expandable_segments` allocator, flash-attn 2 if a wheel exists.
- QLoRA is **not** the fallback here: 91% of the weights are fused 3-D expert tensors, which 4-bit
  `nn.Linear` quantization does not touch. Lowering `--max-len` is.
- Expected peak ≈ 28–32 GiB. The smoke cell measures it on the **longest** rows, so the full run cannot exceed it.

## 1. Setup finished?

In [ ]:
!tail -8 ~/wisp/setup.log; ls -l ~/wisp/SETUP_DONE

In [ ]:
!~/wisp/run.sh check

## 2. Upload the data
Upload `mellum-r1-data.zip` into **wisp/uploads** using the file browser. Compare the sha256 that `check` prints
with `training/mellum-dataset-r1-20261014/upload-zip.json`. `unpack` checks every file against the manifest
inside the zip.

In [ ]:
!~/wisp/run.sh check | grep 'data zip'
!~/wisp/run.sh unpack

## 3. Preflight (tokenizer only, no GPU)
It must print `PREFLIGHT OK`. It checks: the template sha matches the one the data was rendered with, no row is over `--max-len`, every completion ends in `<|im_end|>`, and decoding prompt+completion ids returns the exact text.

In [ ]:
!~/wisp/run.sh preflight

## 4. Smoke run: peak memory and speed on the 8 longest rows
| result | do |
|---|---|
| `MEMORY OK` | train (cell 5) |
| `MEMORY TIGHT` or CUDA OOM | re-run with a lower cap: `!~/wisp/run.sh smoke 10240`, then train with the same cap: `!~/wisp/run.sh train "" 10240` |
| `WARNING: model ignored logits_to_keep` | memory is higher than planned; it still trains if `MEMORY OK` |

The estimate line gives the full-run time.

In [ ]:
!~/wisp/run.sh smoke

## 5. Train (runs in the background; safe to close the tab)
It evaluates and checkpoints every 50 optimizer steps, keeps the best adapter by dev loss in `best/`, and keeps the last 3 steps. If it stops, resume with `!~/wisp/run.sh train ~/wisp/mellum-r1-lora/step-NNNNN`.

In [ ]:
!~/wisp/run.sh train

In [ ]:
# re-run this cell to check progress
!~/wisp/run.sh status

In [ ]:
import json, pathlib
log = [json.loads(l) for l in pathlib.Path.home().joinpath("wisp/mellum-r1-lora/train.log.jsonl").read_text().splitlines()]
steps = [r for r in log if "step" in r and "loss" in r]
evals = [r for r in log if r.get("event") == "eval"]
for i in range(0, len(steps), max(1, len(steps) // 12)):
    r = steps[i]; print(f"step {r['step']:>4}/{r['of']}  loss {r['loss']:.3f}  acc {r['token_acc']:.3f}  peak {r['peak_gib']}GiB  eta {r['eta_min']}m")
for e in evals: print("eval", e["step"], e["dev_loss"], e["dev_token_acc"], e["dev_exact_rows"])

## 6. Merge the best adapter into bf16 weights
It checks that the merged model gives the same dev-row loss as base+adapter before saving (~24GB).

In [ ]:
!~/wisp/run.sh merge

## 7. Package

In [ ]:
!~/wisp/run.sh package

## 8. Back on the Mac (not in this notebook)
1. Download `wisp/mellum-r1-adapter.zip`, plus the merged `*.safetensors`, configs and `SHA256SUMS` from
   `wisp/Mellum2.1-12B-A2.5B-Thinking-Wisp-R1-merged/`. Verify with `shasum -a 256 -c SHA256SUMS`.
2. Quantize the merged folder with oMLX to **oQ5e**, the same as the baseline, so the gate compares weights and not
   quantization. Name it `Mellum2.1-12B-A2.5B-Thinking-Wisp-R1-oQ5e`.
3. Quit Wisp, then run the gate against the baseline (3 trials, narration-off thinking):
   `python scripts/run_toolbench.py --models Mellum2.1-12B-A2.5B-Thinking-Wisp-R1-oQ5e --split gate --trials 3 --thinking narration-off --exclusive`
4. Promote only if every plan threshold holds: pass ≥92%, unsafe 0, runaway ≤1, no-tool ≥19/21, leaks 0,
   reasoning p90 ≤2,800 chars, median ≤4.7s.